### Environment Setup:
###### Installs critical deep learning dependencies and imports the precise Hugging Face models, memory-efficient PEFT adapters, and audio metrics required for the fine-tuning workflow.

In [ ]:
!pip install torch torchaudio transformers datasets peft bitsandbytes jiwer gradio accelerate
!pip install --upgrade transformers peft torchao accelerate

In [ ]:
import os
import torch
import gradio as gr
import torchaudio

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
from datasets import load_dataset, Audio
from dataclasses import dataclass
from jiwer import wer, cer
from peft import LoraConfig,PeftModel, PeftConfig, get_peft_model
from typing import Any, Dict, List, Union

### Configuration:
######Defines core model checkpoints, dataset sources, and memory-optimized training hyperparameters necessary for the VRAM-constrained fine-tuning pipeline.

In [ ]:
MODEL_NAME = "openai/whisper-small"
DATASET_NAME = "gmenon/slt-lyrics-audio"
OUTPUT_DIR = "./whisper-lora-autolyrics"
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 1e-4
MAX_STEPS = 100
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### Data Pipeline:
######Ingests audio, enforces the strict 16kHz resampling requirement, maps waveforms to spectrograms, and dynamically pads variable-length batches for training.

In [ ]:
print("\n--- Loading Dataset ---")
raw_dataset = load_dataset(DATASET_NAME)

raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="english", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    target_text = batch.get("text", batch.get("lyrics", ""))
    batch["labels"] = processor.tokenizer(target_text).input_ids
    return batch

print("Preprocessing dataset (Resampling & Tokenizing)...")
train_split = raw_dataset["train"].select(range(min(150, len(raw_dataset["train"]))))
test_split = raw_dataset["test"].select(range(min(12, len(raw_dataset["test"])))) if "test" in raw_dataset else train_split

train_dataset = train_split.map(prepare_dataset, remove_columns=train_split.column_names)
eval_dataset = test_split.map(prepare_dataset, remove_columns=test_split.column_names)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


--- Loading Dataset ---
Preprocessing dataset (Resampling & Tokenizing)...


### Model Initialization & PEFT:
###### Quantizes the base Whisper model into 8-bit precision to save VRAM and injects low-rank adapters (LoRA) specifically into the attention layers to enable parameter-efficient training.

In [ ]:
from transformers import BitsAndBytesConfig

print("\n--- Initializing Model & LoRA Config ---")

quantization_config = None
if device == "cuda":
    quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto" if device == "cuda" else None
)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


--- Initializing Model & LoRA Config ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


### Model Training and Serialization:
###### Re-initializes the unquantized model wrapper, configures gradient checkpointing, sets up the Seq2Seq execution loop with custom Word Error Rate (WER) evaluations, and executes the fine-tuning process before saving the trained LoRA adapters.

In [ ]:
print("\n--- Loading Model and Applying LoRA ---")

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)

model.config.use_cache = False
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    )

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


print("\n--- Defining Evaluation Metric ---")
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer_score = 100 * wer(label_str, pred_str)
    return {"wer": wer_score}


print("\n--- Setting Up Trainer ---")

train_dataset = train_dataset.select_columns(["input_features", "labels"])
eval_dataset = eval_dataset.select_columns(["input_features", "labels"])

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    max_steps=MAX_STEPS,
    gradient_checkpointing=True,
    fp16=True if device == "cuda" else False,
    eval_strategy="steps",
    per_device_eval_batch_size=BATCH_SIZE,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=10,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    label_names=["labels"],
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

print("\n--- Starting Training ---")
trainer.train()

print("\n--- Saving the Finetuned Adapter ---")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model and processor successfully saved to {OUTPUT_DIR}")


--- Loading Model and Applying LoRA ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429

--- Defining Evaluation Metric ---

--- Setting Up Trainer ---

--- Starting Training ---


Step,Training Loss,Validation Loss,Wer
50,4.323912,2.092016,15000
100,3.450784,1.683266,15000



--- Saving the Finetuned Adapter ---
Model and processor successfully saved to ./whisper-lora-autolyrics


### Performance Evaluation:
###### Compares the zero-shot base model against the fine-tuned LoRA model on the validation dataset, calculating explicit WER/CER metrics and relative error reductions.

In [ ]:
print("\n--- Running Performance Evaluation ---")

def evaluate_predictions(base_model_name, adapter_path, eval_data):
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name).to(device)

    peft_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)
    peft_model = PeftModel.from_pretrained(peft_model, adapter_path).to(device)

    references = []
    base_preds = []
    lora_preds = []

    for sample in eval_data.select(range(min(5, len(eval_data)))):
        input_feats = torch.tensor([sample["input_features"]]).to(device)

        ref_ids = [i for i in sample["labels"] if i != -100]
        ref_text = processor.tokenizer.decode(ref_ids, skip_special_tokens=True)
        references.append(ref_text)

        with torch.no_grad():
            base_gen = base_model.generate(input_features=input_feats)
            base_pred = processor.tokenizer.decode(base_gen[0], skip_special_tokens=True)
            base_preds.append(base_pred)

            lora_gen = peft_model.generate(input_features=input_feats)
            lora_pred = processor.tokenizer.decode(lora_gen[0], skip_special_tokens=True)
            lora_preds.append(lora_pred)
    base_wer = wer(references, base_preds)
    lora_wer = wer(references, lora_preds)
    base_cer = cer(references, base_preds)
    lora_cer = cer(references, lora_preds)

    print("Results")
    print(f"Base Model ({base_model_name}) :- WER: {base_wer:.4f} %| CER: {base_cer:.4f} %")
    print(f"LoRA Fine-Tuned Model     :- WER: {lora_wer:.4f} %| CER: {lora_cer:.4f} %")
    if base_wer > 0:
        improvement = ((base_wer - lora_wer) / base_wer) * 100
        print(f"Relative WER Reduction: {improvement:.2f}%")


evaluate_predictions(MODEL_NAME, OUTPUT_DIR, eval_dataset)


--- Running Performance Evaluation ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Results
Base Model (openai/whisper-small) :- WER: 28.0000 %| CER: 127.0000 %
LoRA Fine-Tuned Model     :- WER: 5.0000 %| CER: 8.0000 %
Relative WER Reduction: 82.14%


### Interactive Deployment:
######Builds a live Gradio web application that loads the baseline and LoRA-adapted models, handles runtime audio transformations, and runs parallel comparative inference.

In [ ]:
!pip install librosa
import torchaudio
# Test MP3 loading directly
info = torchaudio.info("/path/to/any/test.mp3")
print(info)

AttributeError: module 'torchaudio' has no attribute 'info'

In [ ]:

print("\n--- Deploying Clean Live-Model Gradio Web Demo ---")
import torch
import torchaudio
import gradio as gr
from transformers import WhisperForConditionalGeneration

inference_model = model.to(device)
inference_model.eval()
inference_model.config.use_cache = True

print("Loading Baseline Model for comparison...")
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto" if device == "cuda" else None
)
baseline_model.eval()
baseline_model.config.use_cache = True

def transcribe_singing(audio_file):
    if audio_file is None:
        return "No audio file provided.", "No audio file provided."
    audio_file_path = audio_file.name
    import librosa
    import numpy as np
    speech_array, sampling_rate = librosa.load(audio_file_path, sr=16000, mono=True)
    speech_array = torch.tensor(speech_array).unsqueeze(0)  # add channel dim

    input_features = processor.feature_extractor(
        speech_array.squeeze().numpy(), sampling_rate=16000
    ).input_features[0]

    input_tensor = torch.tensor([input_features], dtype=inference_model.dtype).to(device)

    with torch.no_grad():
        predicted_ids_tuned = inference_model.generate(
            input_features=input_tensor,
            language="english",
            task="transcribe",
            max_new_tokens=225,
            num_beams=3,
        )

        predicted_ids_base = baseline_model.generate(
            input_features=input_tensor,
            language="english",
            task="transcribe",
            max_new_tokens=225,
            num_beams=3,
        )

    transcription_tuned = processor.batch_decode(predicted_ids_tuned, skip_special_tokens=True)[0]
    transcription_base = processor.batch_decode(predicted_ids_base, skip_special_tokens=True)[0]

    return transcription_tuned, transcription_base

custom_css = """
body, .gradio-container {
    background-color: #0b0c10 !important;
    color: #e5e7eb !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
.studio-title h1 {
    background: linear-gradient(90deg, #a855f7 0%, #06b6d4 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-weight: 800 !important;
    letter-spacing: -0.05em;
    text-shadow: 0 0 30px rgba(168, 85, 247, 0.2);
}
.card-glow {
    background: #111217 !important;
    border: 1px solid #1f2937 !important;
    border-radius: 12px !important;
    padding: 20px !important;
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.4) !important;
}
.tuned-box {
    border-left: 4px solid #a855f7 !important;
}
.base-box {
    border-left: 4px solid #4b5563 !important;
}
.telemetry-row {
    margin-top: 20px !important;
}
footer {
    display: none !important;
}
"""

force_dark_js = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, js=force_dark_js) as demo:

    with gr.Row(elem_classes=["studio-header"]):
        with gr.Column(scale=4):
            gr.Markdown(
                """
                # 🎵 AutoLyrics
                ### By: Ayush Bansal, Bhumika, Lalit Deshmane
                """,
                elem_classes=["studio-title"]
            )

    gr.HTML("<hr style='border-color: #1f2937; margin-bottom: 25px;'>")

    with gr.Row():

        with gr.Column(scale=1, elem_classes=["card-glow"]):
            gr.Markdown("### 🎤 Audio Ingestion")
            audio_input = gr.File(
    label="Upload singing audio or music clip here:",
    file_types=[".mp3", ".wav", ".ogg", ".flac", ".m4a", ".mpeg", ".mp4"],
    interactive=True
)

            gr.HTML("<div style='margin-top: 15px;'></div>")

            # Interactive Triggers
            submit_btn = gr.Button(
                "Submit",
                variant="primary",
                size="sm"
            )
            clear_btn = gr.Button(
                "Reset Interface/ Clear",
                variant="secondary",
                size="sm"
            )

        with gr.Column(scale=2):
            with gr.Row():
                with gr.Column(elem_classes=["card-glow", "tuned-box"]):
                    gr.Markdown("#### AutoLyrics Fine-Tuned Transcription(`LoRA Fine-Tuned`)")
                    tuned_output = gr.Textbox(
                        label="",
                        placeholder="Acoustically aligned lyrics tracking matrix will appear here...",
                        lines=2,
                        show_copy_button=True,
                        container=False
                    )

                with gr.Column(elem_classes=["card-glow", "base-box"]):
                    gr.Markdown("#### Baseline (Un-Tuned) Transcription (`Whisper-Small`)")
                    base_output = gr.Textbox(
                        label="",
                        placeholder="Standard conversational voice stream transcription will appear here...",
                        lines=2,
                        show_copy_button=True,
                        container=False
                    )

    with gr.Row(elem_classes=["card-glow", "telemetry-row"]):
        gr.Markdown(
            """
            ##### 💻 Technical Blueprint:
            * **Acoustic Normalization Stage:** Automatic Mono Downmixing & Real-time 16kHz Resampling Matrix
            * **Neural Adapter Layer:** Low-Rank Adaptation (LoRA Target: $Q, V$ Projections | Rank: 32 | Scaling Alpha: 64)
            * **Sequence Search Space:** Multi-path Autoregressive Beam Search (Width: 3)
            """
        )

    submit_btn.click(
        fn=transcribe_singing,
        inputs=[audio_input],
        outputs=[tuned_output, base_output]
    )

    clear_btn.click(
        fn=lambda: (None, "", ""),
        inputs=None,
        outputs=[audio_input, tuned_output, base_output]
    )

demo.launch(share=True)


--- Deploying Clean Live-Model Gradio Web Demo ---
Loading Baseline Model for comparison...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

/tmp/ipykernel_12272/750523377.py:101: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, js=force_dark_js) as demo:
/tmp/ipykernel_12272/750523377.py:101: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, js=force_dark_js) as demo:
/tmp/ipykernel_12272/750523377.py:101: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, js=force_dark_js) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7797e56d40b0512b42.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
